<a href="https://colab.research.google.com/github/hig3r/mva/blob/2026/E05/mva_2026_E05_03_multicolinearity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# mva_2026_E05_03_multicolinearity 多重共線性の実験

In [ ]:
import numpy as np # 数値計算ライブラリ
import pandas as pd # データ解析ライブラリ
import matplotlib.pyplot as plt # グラフ描画ライブラリ
%matplotlib inline

from scipy import stats

from sklearn.linear_model import LinearRegression # scikit-learn の線形重回帰ライブラリ
from sklearn.model_selection import train_test_split # scikit-learn の訓練データとテストデータ分割ライブラリ
from sklearn.metrics import root_mean_squared_error, r2_score

#%pip install japanize_matplotlib
#import japanize_matplotlib # 日本語フォント

これまで，
* scipy.stats で多変量正規分布に従うデータを自分で生成すること
* 与えられたデータを線形重回帰モデルに当てはめること
はしばしばやってきました

今回は，与えられたデータでなく，自分で生成したデータを線形重回帰モデルに当てはめて失敗してみます

自分で生成したのだから，回帰係数はわかってて，推定する必要なんかないのに，って話ですが，推定がいつ成功していつ失敗するかを知るための実験です．それを知るためには，モデル，答がわかってる自家製データでやる必要があります．

In [ ]:
m = 10 # 実験の繰り返し回数
n = 1000 # 生成するサンプルのサイズ

# 仏のみ知る真の回帰係数
beta0=4
beta1=3
beta2=-5
sig=2 # ノイズの大きさ

## 説明変数を多変量正規分布からランダムに作る

In [ ]:
# 説明変数は，人が決めてもいいのだが，多変量正規分布からランダムに作る
mean = [5, 9]
sigma = [[4, 2],
          [2, 9]]
rvxy=stats.multivariate_normal(mean=mean, cov=sigma)

In [ ]:
for i in range(0,m,1): # m 回繰り返す
  X = pd.DataFrame(rvxy.rvs(size=n), columns=['x1','x2'])

  # 誤差項（ノイズ）を生成して y を計算します
  rvz=stats.norm(loc=0, scale=1)
  epsilon=sig*rvz.rvs(size=n)
  y = beta0 + beta1 * X['x1'] + beta2 * X['x2'] + epsilon

  model=LinearRegression()
  model.fit(X,y)
  print("回帰係数, R^2")
  print(model.coef_, model.score(X,y))

## 多重共線性のある場合

In [ ]:
  rvx1=stats.norm(loc=mean[0],scale=np.sqrt(sigma[0][0]))

In [ ]:
for i in range(0,m,1):
  # 説明変数x1を正規分布からランダムに作り，x2はその定数倍とする
  x1=rvx1.rvs(size=n)
  x2=x1* np.sqrt(sigma[1][1]/sigma[0][0]) # 正比例で作る　多重共線性を持つ
  X = pd.DataFrame({'x1':x1, 'x2':x2})

  # 以下は同じです
  # 誤差項（ノイズ）を生成して y を計算します
  rvz=stats.norm(loc=0, scale=1)
  epsilon=sig*rvz.rvs(size=n)
  y = beta0 + beta1 * X['x1'] + beta2 * X['x2'] + epsilon

  model=LinearRegression()
  model.fit(X,y)
  print("回帰係数, R^2")
  print(model.coef_, model.score(X,y))

上のように，警告やエラーなしに，仏の定めたのと全然異なる（しかし毎回ほぼ同じ）回帰係数の推定値が返される．
この結果を信用してはいけない．多重共線性があるので，もともと，意味のある$\beta_1,\beta_2$ の区別がない．